========================================
### GOLD LAYER – CUSTOMER METRICS
----------------------------------------
###Source : Silver Orders + Silver Customers
###Target : Delta Gold Aggregate Table
###Grain  : One row per customer
###Load   : Full Refresh (Overwrite)
###Purpose: Customer-level KPIs & insights
========================================


### READ SILVER TABLES

In [0]:
orders = spark.table("olist_silver_orders")
customers = spark.table("olist_silver_customers")
order_items = spark.table("olist_silver_order_items")


### JOIN

In [0]:
fact = (
    orders
    .join(customers, "customer_id", "left")
    .join(order_items, "order_id", "left")
)


### CUSTOMER LEVEL AGGREATION

In [0]:
from pyspark.sql.functions import countDistinct, sum, avg, max

# Aggregate order data for each customer by city and state:
gold_customer = (
    fact
    .groupBy("customer_id", "customer_city", "customer_state")
    .agg(
        countDistinct("order_id").alias("total_orders"),         # Number of unique orders per customer
        sum("price").alias("total_revenue"),                     # Total revenue per customer
        avg("price").alias("avg_order_value"),                   # Average order value per customer
        max("order_purchase_timestamp").alias("last_order_ts")   # Most recent order timestamp per customer
    )
)

### WRITE TO GOLD

In [0]:
(
    gold_customer
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("olist_gold_customer_metrics")
)


### OPTIMIZE

In [0]:
%sql
OPTIMIZE olist_gold_customer_metrics
ZORDER BY (customer_state, customer_city);


path,metrics
abfss://unity-catalog-storage@dbstoragetic7vxegr5zes.dfs.core.windows.net/7405614582366842/__unitystorage/catalogs/4444e7c2-d2e1-4e77-b5c3-b026efd7d282/tables/06741598-90c0-4adb-bda8-9dd7dc55a556,"List(1, 4, List(2978586, 2978586, 2978586.0, 1, 2978586), List(776487, 790747, 781574.5, 4, 3126298), 0, List(minCubeSize(107374182400), List(0, 0), List(4, 3126298), 0, List(4, 3126298), 1, null), null, 0, 1, 4, 0, false, 0, 0, 1769886388512, 1769886390554, 4, 1, null, List(0, 0), null, 7, 7, 621, 0, null)"


In [0]:
%sql
SELECT COUNT(*) FROM olist_gold_customer_metrics;
SELECT * FROM olist_gold_customer_metrics LIMIT 10;


customer_id,customer_city,customer_state,total_orders,total_revenue,avg_order_value,last_order_ts
25c6702d4b98e41d6bc8ebfa53bafab5,ALTO PARAISO DE GOIAS,GO,1,49.98,24.99,2018-02-09T00:50:48Z
43310f2ccac055857c45f0158299a2e8,SAO PAULO,SP,1,46.0,23.0,2017-05-01T13:11:44Z
3df001627683ff649e8df47730d2b52c,SAO PAULO,SP,1,98.0,49.0,2017-11-26T15:51:16Z
264a54d065d7a43fa179aac655c854e6,SENADOR POMPEU,CE,1,36.0,18.0,2018-06-18T19:51:14Z
1ceacda3f304066791cb6ad623ee3e90,PRAIA GRANDE,SP,1,139.8,69.9,2017-09-26T23:40:53Z
356e49e06716cab5d634234a371dba70,GOIANIA,GO,1,699.98,349.99,2017-01-20T15:08:05Z
ae8ed9c83d32613ae9f7e6c72443cdbb,RIBEIRAO PIRES,SP,1,91.8,45.9,2017-03-01T10:46:15Z
76cff9a9995b25f21d51697f49f87652,SAO PAULO,SP,1,894.0,447.0,2017-04-21T17:49:46Z
9db60441422191db18225cf1e58e561f,CORUMBA,MS,1,199.98,99.99,2018-06-10T12:51:50Z
67a28655b804d6a418964f3dd4e214bc,VILA VELHA,ES,1,392.0,49.0,2018-01-23T18:47:39Z
